# Objetivo
Generar samples y obtener una idea del escalado temporal, todo aprovechando de usar tecnicas de batching programadas; de esta manera generando datos para los modelos de ML clasificativos.

In [ ]:
import numpy as np
import pandas as pd
import time
import psutil
import pickle
from scipy.stats import qmc
from lib.oracle import OracleExecutor  # assumes your OracleExecutor is in oracle_wrapper.py


epsilon = 0.001
vev = 246

m12_max = epsilon * vev**2 #**2
# Define the ranges for each parameter:
# [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
# TODO
# 300 done 7 batches
# 290 
# 280 m_phi_base, m12_2_base = 280, 7.83990900e+00
# 270 m_phi_base, m12_2_base = 270, 7.83990900e+00
# 260
m_phi_base, m12_2_base = 260, 6.75990900


param_bounds = np.array([
    [m_phi_base - 3* epsilon, m_phi_base + 3*epsilon],    # m_phi (GeV)
    [300 - epsilon, 300 + epsilon],    # m_A   (GeV)
    [1.0 - epsilon, 1.0],        # sin(b - a)
    [10000.0 - epsilon, 10000.0 + epsilon],        # tan(beta)
    [0.1 - epsilon, 0.1 + epsilon],# lambda6
    [0, epsilon],# lambda7
    [m12_2_base - epsilon, m12_2_base + epsilon]  # m12^2, agregado para que sea pequeño
])

def generate_params(batch_idx, param_bounds, batch_size):
    sampler = qmc.LatinHypercube(d=7)
    unit_sample = sampler.random(n=batch_size)
    param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

    # fijando parametros
    param_list[:, 1] = 300.0 # m_A 
    param_list[:, 2] = 1 # sin(b-a)
    param_list[:, 3] = 10000 # tan_beta
    param_list[:, 4] = 0.1 # lambda6
    param_list[:, 5] = 0.0 # lambda7

    return param_list


def get_parameters_from_points():
    """
    By having a pre existing combination of m_phi and m_12^2, generates a list of params bounds and paramlist
    This its done using the latin Hypercube in order to maximize variations
    """

# Prepare executor
executor = OracleExecutor(nthreads=4)



# Runs

In [7]:
import os
import glob
import pickle
import psutil
import time
import numpy as np
from scipy.stats import qmc

batch_size = 15_000
outdir = "data_batches"
max_merge_size_mb = 30

os.makedirs(outdir, exist_ok=True)

existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
batch_idx

3

In [ ]:
def multiple_runs(n_runs, batch_size, param_bounds):
    
    time_per_points = 415.7 / 15_000
    pred_mins = n_runs * batch_size * time_per_points / 60 # mins
    print("Predicted \tmins:", pred_mins, "[mins]")
    print("          \thours:", pred_mins/60, "[hr]")

    for jth_run in range(n_runs):
        # ------------------------
        # Determine next batch index
        # ------------------------
        existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
        batch_idx = len(existing_batches) + 1
        print(batch_idx)

        # ------------------------
        # Generate Latin-Hypercube sample
        # ------------------------
        param_list = generate_params(batch_idx, param_bounds, batch_size)

        # ------------------------
        # Run your oracle / executor
        # ------------------------
        t0 = time.perf_counter()
        results = executor.map(param_list.tolist(), use_threads=True)
        t1 = time.perf_counter()

        # ------------------------
        # Save this batch
        # ------------------------
        outfile = f"{outdir}/batch_{batch_idx}_{m_phi_base}.pkl"
        with open(outfile, "wb") as f:
            pickle.dump({"params": param_list, "results": results}, f)

        print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")
    return results


In [5]:
multiple_runs(3, batch_size)

Predicted 	mins: 20.785 [mins]
          	hours: 0.34641666666666665 [hr]
21
Batch 21 saved (15000 points) in 475.6s → data_batches/batch_21_270.pkl
2
Batch 2 saved (15000 points) in 476.6s → data_batches/batch_2_270.pkl
3


KeyboardInterrupt: 

In [8]:
multiple_runs(3, batch_size)

Predicted 	mins: 20.785 [mins]
          	hours: 0.34641666666666665 [hr]
3
Batch 3 saved (15000 points) in 284.9s → data_batches/batch_3_260.pkl
4
Batch 4 saved (15000 points) in 274.8s → data_batches/batch_4_260.pkl
5
Batch 5 saved (15000 points) in 265.6s → data_batches/batch_5_260.pkl


[{'positivity_ok': 0,
  'unitarity_ok': 0,
  'perturbativity_ok': 0,
  'w_h2_bb': 0.0,
  'w_h2_tautau': 0.0,
  'w_h2_uu': 0.0,
  'w_h2_du': 0.0,
  'w_h2_ln': 0.0,
  'w_h2_vv': [1.561054828652782e-10, 0.0, 0.0],
  'w_h2_gaga': 1.561054828652782e-10,
  'w_h2_Zga': 9.819155356043287e-11,
  'w_h2_gg': 0.0,
  'w_h2_hh': 1.58034536781869e-32,
  'w_total_h2': 2.542970364257111e-10,
  'w_total_top': 1.338069402459526,
  'branching_ratio_h2_gaga': 0.6138706335686377,
  'lambda1': -1715.986500205799,
  'lambda2': 0.2577337926164514,
  'lambda3': 0.9966765549898081,
  'lambda4': -0.3694777999652601,
  'lambda5': -0.3694777999652601,
  'lambda6': 0.1,
  'lambda7': 0.0},
 {'positivity_ok': 1,
  'unitarity_ok': 1,
  'perturbativity_ok': 0,
  'w_h2_bb': 0.0,
  'w_h2_tautau': 0.0,
  'w_h2_uu': 0.0,
  'w_h2_du': 0.0,
  'w_h2_ln': 0.0,
  'w_h2_vv': [3.08917908220271e-10, 0.0, 0.0],
  'w_h2_gaga': 3.08917908220271e-10,
  'w_h2_Zga': 1.943093851372216e-10,
  'w_h2_gg': 0.0,
  'w_h2_hh': 1.578081750440986e

In [19]:
print(param_list[:5])
print(len(param_list))

print(results[:5])
print(len(results))

# all pickles are made like:
#    pickle.dump({"params": param_list, "results": results}, f)


[[ 2.90972554e+02  2.42671492e+02  9.99906968e-01  1.83445228e+03
   3.58254116e-05  9.63008916e-05  5.10090209e+00]
 [ 3.90106060e+02  3.52647880e+02  9.99982419e-01  9.86387544e+03
   6.51931101e-05  9.44609250e-05  3.14377118e+00]
 [ 1.45053133e+02  3.97831745e+02  9.99910765e-01  3.91033007e+03
  -5.33026162e-05  9.01694304e-05  3.64750044e+00]
 [ 2.24301321e+02  3.20172639e+02  9.99942371e-01  2.13994012e+03
  -7.65944688e-05  1.70070492e-05  2.27629995e+00]
 [ 1.57675251e+02  2.88391895e+02  9.99979244e-01  5.63763949e+03
  -8.95021400e-05 -5.67457529e-05  2.66907545e+00]]
15000
[{'positivity_ok': None, 'unitarity_ok': None, 'perturbativity_ok': None, 'w_h2_bb': None, 'w_h2_tautau': None, 'w_h2_uu': None, 'w_h2_du': None, 'w_h2_ln': None, 'w_h2_vv': [None, None, None], 'w_h2_gaga': None, 'w_h2_Zga': None, 'w_h2_gg': None, 'w_h2_hh': None, 'w_total_h2': None, 'w_total_top': None, 'branching_ratio_h2_gaga': None, 'lambda1': None, 'lambda2': None, 'lambda3': None, 'lambda4': None, '

In [20]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



5
Batch 5 saved (15000 points) in 863.3s → data_batches/batch_5.pkl


In [ ]:
results

In [ ]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=True)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



1



# Testing Speed

In [ ]:
# Define the sampling sizes
sample_sizes = [1, 10, 100, 1_000, 10_000]

for n in sample_sizes:
    # Generate Latin Hypercube samples in [0,1]^7, then scale
    sampler = qmc.LatinHypercube(d=7)
    sample_unit = sampler.random(n)
    param_list = qmc.scale(sample_unit, param_bounds[:,0], param_bounds[:,1])
    
    # Measure memory before run
    process = psutil.Process()
    mem_before = process.memory_info().rss
    
    # Run and time
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=False)
    t1 = time.perf_counter()
    
    mem_after = process.memory_info().rss
    delta_mem = (mem_after - mem_before) / (1024**2)  # in MB
    
    # Save raw results for this batch
    with open(f"oracle_results_{n}.pkl", "wb") as f:
        pickle.dump(results, f)
    
    # Record performance
    perf_records.append({
        "n_points": n,
        "time_sec": t1 - t0,
        "mem_delta_MB": delta_mem
    })
    print(f"Completed batch {n}: time={t1-t0:.2f}s, memory Δ={delta_mem:.1f}MB")

# Save performance table
df_perf = pd.DataFrame(perf_records)
df_perf.to_csv("performance_scaling.csv", index=False)



# Merging

In [ ]:
# ------------------------
# Merge old batches if they exceed size threshold
# ------------------------
def merge_batches(folder, batch_prefix="batch_", merged_prefix="merged_", max_size_mb=30):
    # Count existing merged files to avoid overwrite
    existing_merged = sorted(glob.glob(f"{folder}/{merged_prefix}*.pkl"))
    merge_idx = len(existing_merged) + 1
    
    # Only consider raw batch files
    batch_files = sorted(glob.glob(f"{folder}/{batch_prefix}*.pkl"))
    acc_size = 0
    group = []

    for fp in batch_files:
        fsize = os.path.getsize(fp)
        if (acc_size + fsize) / (1024**2) > max_size_mb and group:
            # Merge current group
            merged_data = []
            for gfp in group:
                with open(gfp, "rb") as gf:
                    merged_data.append(pickle.load(gf))
                os.remove(gfp)
            mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
            with open(mout, "wb") as mf:
                pickle.dump(merged_data, mf)
            print(f"Merged {len(group)} batches into {mout}")
            merge_idx += 1
            group, acc_size = [], 0

        group.append(fp)
        acc_size += fsize

    # Merge any remaining files
    if group:
        merged_data = []
        for gfp in group:
            with open(gfp, "rb") as gf:
                merged_data.append(pickle.load(gf))
            os.remove(gfp)
        mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
        with open(mout, "wb") as mf:
            pickle.dump(merged_data, mf)
        print(f"Merged {len(group)} batches into {mout}")
    return merged_data

# Call merge
merged_data = merge_batches(outdir)
merged_data

Merged 2 batches into data_batches/merged_9.pkl


[{'params': array([[ 1.65106260e+02,  1.70883284e+02,  9.63427920e-01,
           2.71242992e+03,  6.53763371e-03, -4.28466529e-03,
           1.43103852e+00],
         [ 3.17185963e+02,  3.09266503e+02,  9.79986220e-01,
           5.52912404e+03, -3.30762866e-03, -2.66594634e-03,
           2.24380465e+00],
         [ 1.39978179e+02,  2.91122058e+02,  9.75932979e-01,
           7.56212901e+03, -1.04069105e-03,  9.94753733e-03,
           1.29141029e-01],
         [ 3.32462488e+02,  1.94804793e+02,  9.59009631e-01,
           4.50933045e+03,  7.85504201e-03, -3.08245371e-04,
           1.24778848e+00],
         [ 2.62605588e+02,  2.52211276e+02,  9.91241966e-01,
           6.49192503e+03, -1.52469795e-03,  1.24466338e-03,
           2.37112012e+00],
         [ 1.45105998e+02,  3.59253506e+02,  9.76235705e-01,
           4.25981235e+03, -5.21246992e-03, -1.92826563e-03,
           1.52882523e+00],
         [ 2.16606118e+02,  4.91637639e+02,  9.72080059e-01,
           5.85320136e+03,  4

In [27]:
merged_data[0]

{'params': array([[ 1.65106260e+02,  1.70883284e+02,  9.63427920e-01,
          2.71242992e+03,  6.53763371e-03, -4.28466529e-03,
          1.43103852e+00],
        [ 3.17185963e+02,  3.09266503e+02,  9.79986220e-01,
          5.52912404e+03, -3.30762866e-03, -2.66594634e-03,
          2.24380465e+00],
        [ 1.39978179e+02,  2.91122058e+02,  9.75932979e-01,
          7.56212901e+03, -1.04069105e-03,  9.94753733e-03,
          1.29141029e-01],
        [ 3.32462488e+02,  1.94804793e+02,  9.59009631e-01,
          4.50933045e+03,  7.85504201e-03, -3.08245371e-04,
          1.24778848e+00],
        [ 2.62605588e+02,  2.52211276e+02,  9.91241966e-01,
          6.49192503e+03, -1.52469795e-03,  1.24466338e-03,
          2.37112012e+00],
        [ 1.45105998e+02,  3.59253506e+02,  9.76235705e-01,
          4.25981235e+03, -5.21246992e-03, -1.92826563e-03,
          1.52882523e+00],
        [ 2.16606118e+02,  4.91637639e+02,  9.72080059e-01,
          5.85320136e+03,  4.09782222e-03, -8.35